# Phase 2 legal NER · word vs. subword BIO

Both runs start from the same Phase 1 decoder and use the same BPE input windows, document split, BiLSTM, CRF, and optimizer settings. The subword run predicts one BIO tag per BPE token. The word run averages the decoder's vectors for the pieces of each word or punctuation unit, then predicts one BIO tag per unit. Word boundaries are used to align windows in both runs so that the decoder sees identical inputs.

**Run in Google Colab with a GPU.** Put your Phase 1 decoder checkpoint (`best.pt`), `bpe_40k.json`, and `a1_tokenizer.py` in My Drive, or edit their paths in the configuration cell. The notebook mounts Drive automatically. Add `HF_TOKEN` in Colab Secrets, enable notebook access, and use a token from an account with access to `Exploration-Lab/CS-781-Capstone`. The data cell downloads the Phase 2 training folds directly from Hugging Face.

The default runs both variants and prints strict document-level span F1 on the original three-document validation split. Each run saves its best checkpoint locally. The original decoder, offset encoder, BiLSTM, and CRF are reused from [the Phase 2 notebook](https://github.com/yugan12345/CS781-Capstone-2026/blob/24e8794/phase-2/exp1_full_ft_bio_fixed.ipynb).

## 1. Configure the run

In [15]:
import os, sys, re, json, math, time, random, bisect, copy, zipfile, glob, shutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("Imports OK")
print("PyTorch:", torch.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Imports OK
PyTorch: 2.14.1+cu130
Device: cuda


In [16]:
@dataclass
class NERConfig:
    # Model (must match Phase 1)
    vocab_size: int      = 40_000
    d_model: int         = 768
    n_layers: int        = 19
    n_query_heads: int   = 12
    n_kv_heads: int      = 4
    d_ff: int            = 2048
    max_seq_len: int     = 512
    dropout: float       = 0.1
    weight_tying: bool   = True
    rope_theta: float    = 10000.0

    # NER
    num_labels: int      = 12
    bio_num_tags: int    = 25      # BIO: 2 * num_labels + 1 (O)
    window_size: int     = 510
    window_stride: int   = 255

    # Training
    lr: float            = 5e-6
    min_lr: float        = 1e-7
    warmup_steps: int    = 200
    num_epochs: int      = 25
    batch_size: int      = 8
    grad_accum: int      = 2       # effective batch = 2*8 = 16
    weight_decay: float  = 0.01
    max_grad_norm: float = 1.0
    use_amp: bool        = True
    seed: int            = 42

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_query_heads
        self.n_groups = self.n_query_heads // self.n_kv_heads

CFG = NERConfig()

OUT       = Path(".")
CKPT_DIR  = OUT / "ner_checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
SUB_DIR   = OUT / "submissions";     SUB_DIR.mkdir(exist_ok=True)

print(f"Config: {CFG.d_model}d x {CFG.n_layers}L, window={CFG.window_size}, stride={CFG.window_stride}")
print(f"Batch={CFG.batch_size} x grad_accum={CFG.grad_accum} = effective {CFG.batch_size * CFG.grad_accum}")

Config: 768d x 19L, window=510, stride=255
Batch=8 x grad_accum=2 = effective 16


In [17]:
from google.colab import drive
import importlib.util

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
PHASE1_CKPT = DRIVE / "best.pt"
TOKENIZER_JSON = DRIVE / "bpe_40k.json"
TOKENIZER_MODULE = DRIVE / "a1_tokenizer.py"

# None downloads the training files from Hugging Face.
DATA_DIR = None
RUN_MODES = ("word",)
EPOCHS = 6
EARLY_STOP_PATIENCE = 2
OVERSAMPLE_RARE_WINDOWS = True
RESULTS_DIR = Path("phase2_word_bio_results")

for path in (PHASE1_CKPT, TOKENIZER_JSON, TOKENIZER_MODULE):
    if not path.is_file():
        raise FileNotFoundError(f"Set the configuration cell to the correct file path: {path}")

spec = importlib.util.spec_from_file_location("a1_tokenizer", TOKENIZER_MODULE)
tokenizer_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = tokenizer_module
spec.loader.exec_module(tokenizer_module)
tokenizer = tokenizer_module.BPETokenizer.load(str(TOKENIZER_JSON))
CFG.vocab_size = tokenizer.vocab_size
CFG.use_amp = CFG.use_amp and DEVICE.type == "cuda"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Load the official training folds

In [18]:
files = ["fold1-text.json", "fold1-gold.json", "fold2-text.json", "fold2-gold.json", "labels.txt"]

if DATA_DIR is None:
    try:
        from huggingface_hub import hf_hub_download
    except ModuleNotFoundError:
        import subprocess

        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import hf_hub_download

    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from google.colab import userdata
        except ImportError:
            pass
        else:
            try:
                hf_token = userdata.get("HF_TOKEN")
            except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
                pass
    if not hf_token:
        raise ValueError("Add HF_TOKEN in Colab Secrets and enable notebook access, or set the HF_TOKEN environment variable.")

    paths = {
        name: Path(hf_hub_download(
            "Exploration-Lab/CS-781-Capstone",
            f"Phase_2_LNER/Train/{name}",
            repo_type="dataset",
            token=hf_token,
        ))
        for name in files
    }
else:
    paths = {name: Path(DATA_DIR) / name for name in files}

def read_json(name):
    return json.loads(paths[name].read_text(encoding="utf-8"))

all_text = {**read_json("fold1-text.json"), **read_json("fold2-text.json")}
all_gold = {**read_json("fold1-gold.json"), **read_json("fold2-gold.json")}
LABELS = paths["labels.txt"].read_text(encoding="utf-8").split()
O_TAG = 0
tag2id = {"O": O_TAG}
for label in LABELS:
    for prefix in ("B-", "I-"):
        tag2id[prefix + label] = len(tag2id)
id2tag = {index: tag for tag, index in tag2id.items()}
CFG.bio_num_tags = len(tag2id)

doc_ids = sorted(all_text)
random.Random(CFG.seed).shuffle(doc_ids)
train_ids, val_ids = doc_ids[:-3], doc_ids[-3:]
print(f"Training documents: {len(train_ids)} | Validation documents: {len(val_ids)}")
print(f"Validation IDs: {val_ids}")

Training documents: 67 | Validation documents: 3
Validation IDs: ['184861797', '108594766', '1348961']


## 3. Reuse the decoder, offset encoder, and CRF

In [19]:
def encode_with_offsets(tokenizer, text):
    token_ids = []
    char_spans = []
    for m in re.finditer(r'\S+', text):
        word = m.group()
        ws = m.start()
        char_byte_bounds = [0]
        for ch in word:
            char_byte_bounds.append(char_byte_bounds[-1] + len(ch.encode('utf-8')))
        bpe_syms = tokenizer._bpe(word)
        SPACE_BYTES = 1
        fb_pos = 0
        for sym in bpe_syms:
            tok_id = tokenizer.token_to_id.get(sym, tokenizer.token_to_id.get('<unk>', 1))
            n_bytes = len(sym)
            fb_end = fb_pos + n_bytes
            wb_s = max(0, fb_pos - SPACE_BYTES)
            wb_e = max(0, fb_end - SPACE_BYTES)
            wb_e = min(wb_e, char_byte_bounds[-1])
            wb_s = min(wb_s, char_byte_bounds[-1])
            if wb_e > wb_s:
                c_s = bisect.bisect_right(char_byte_bounds, wb_s) - 1
                c_e = bisect.bisect_left(char_byte_bounds, wb_e)
                c_s = max(0, c_s)
                c_e = min(len(word), c_e)
                char_spans.append((ws + c_s, ws + c_e))
            else:
                char_spans.append((ws, ws))
            token_ids.append(tok_id)
            fb_pos = fb_end
    return token_ids, char_spans

def get_doc_text(doc_entry):
    if isinstance(doc_entry, str):
        return doc_entry
    if isinstance(doc_entry, dict):
        return doc_entry.get("text", "")
    return str(doc_entry)

print("Offset encoder defined")

Offset encoder defined


In [20]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin, offset=0):
    T = x.size(2)
    c = cos[offset:offset+T].unsqueeze(0).unsqueeze(0)
    s = sin[offset:offset+T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_exp = torch.cat([c.expand(-1,-1,-1,hd), c.expand(-1,-1,-1,hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1,-1,-1,hd), s.expand(-1,-1,-1,hd)], dim=-1)
    return x * c_exp + rotated * s_exp

class GQA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nqh = cfg.n_query_heads
        self.nkh = cfg.n_kv_heads
        self.ng  = cfg.n_groups
        self.hd  = cfg.head_dim
        self.dm  = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.n_query_heads * cfg.head_dim, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads   * cfg.head_dim, bias=False)
        self.o = nn.Linear(cfg.n_query_heads * cfg.head_dim, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)

    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nqh, self.hd).transpose(1,2)
        k = self.k(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        v = self.v(x).view(B, T, self.nkh, self.hd).transpose(1,2)
        q = apply_rope(q, cos, sin, offset)
        k = apply_rope(k, cos, sin, offset)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        if self.ng > 1:
            k = k.unsqueeze(2).expand(B, self.nkh, self.ng, k.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
            v = v.unsqueeze(2).expand(B, self.nkh, self.ng, v.size(2), self.hd).reshape(B, self.nqh, -1, self.hd)
        w = torch.matmul(q, k.transpose(-2,-1)) * self.scale
        if mask is not None: w = w + mask
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1,2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn  = GQA(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn   = SwiGLU(cfg)
    def forward(self, x, cos, sin, mask=None, kv_cache=None, offset=0):
        h, cache = self.attn(self.norm1(x), cos, sin, mask, kv_cache, offset)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x, cache

class LegalLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop     = nn.Dropout(cfg.dropout)
        self.layers   = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm     = RMSNorm(cfg.d_model)
        self.lm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.weight_tying:
            self.lm_head.weight = self.tok_emb.weight
        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len * 2, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _causal_mask(self, T, device, S=None):
        if S is None: S = T
        m = torch.full((T, S), float("-inf"), device=device)
        off = S - T
        for i in range(T):
            m[i, :off+i+1] = 0.0
        return m.unsqueeze(0).unsqueeze(0)

    def forward(self, ids, targets=None, kv_caches=None, offset=0):
        B, T = ids.shape
        x = self.drop(self.tok_emb(ids))
        mask = self._causal_mask(T, ids.device, T + offset)
        new_caches = []
        for i, layer in enumerate(self.layers):
            c = kv_caches[i] if kv_caches else None
            x, nc = layer(x, self.rope_cos, self.rope_sin, mask, c, offset)
            new_caches.append(nc)
        logits = self.lm_head(self.norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size),
                                   targets.view(-1), ignore_index=-1)
        return logits, loss, new_caches

print("Model architecture defined")

Model architecture defined


In [21]:
def load_phase1_backbone(cfg, ckpt_path, device):
    model = LegalLM(cfg)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt.get("model", ckpt)
    model.load_state_dict(state, strict=True)
    model = model.to(device)
    print(f"Phase 1 backbone loaded from {os.path.basename(ckpt_path)}")
    print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
    return model

print("Backbone loader defined")

Backbone loader defined


In [22]:
# ========================================================
# CRITICAL FIX: CRF forward algorithm emission dimension
# OLD BUG: emit = emissions[:, t].unsqueeze(1)  -> wrong dim
# NEW FIX: emit = emissions[:, t].unsqueeze(2)  -> correct dim
# ========================================================

from torch.utils.checkpoint import checkpoint as ckpt_fn

class CRF(nn.Module):
    """Linear-chain CRF with BUG-FIXED forward algorithm."""
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.randn(num_tags, num_tags))
        self.start_transitions = nn.Parameter(torch.randn(num_tags))
        self.end_transitions = nn.Parameter(torch.randn(num_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)

    def forward(self, emissions, tags, mask):
        gold_score = self._score_sentence(emissions, tags, mask)
        forward_score = self._forward_alg(emissions, mask)
        return (forward_score - gold_score).mean()

    def _score_sentence(self, emissions, tags, mask):
        B, T, C = emissions.shape
        score = self.start_transitions[tags[:, 0]] + emissions[:, 0].gather(1, tags[:, 0:1]).squeeze(1)
        for t in range(1, T):
            trans = self.transitions[tags[:, t], tags[:, t-1]]
            emit = emissions[:, t].gather(1, tags[:, t:t+1]).squeeze(1)
            step_score = trans + emit
            score = torch.where(mask[:, t], score + step_score, score)
        last_idx = mask.long().sum(dim=1) - 1
        last_tags = tags.gather(1, last_idx.unsqueeze(1)).squeeze(1)
        score = score + self.end_transitions[last_tags]
        return score

    def _forward_alg(self, emissions, mask):
        B, T, C = emissions.shape
        alpha = self.start_transitions.unsqueeze(0) + emissions[:, 0]
        for t in range(1, T):
            # ============ BUG FIX ============
            emit = emissions[:, t].unsqueeze(2)   # FIXED: was unsqueeze(1)
            # =================================
            trans = self.transitions.unsqueeze(0)
            alpha_exp = alpha.unsqueeze(1) + trans + emit
            new_alpha = torch.logsumexp(alpha_exp, dim=2)
            alpha = torch.where(mask[:, t].unsqueeze(1), new_alpha, alpha)
        alpha = alpha + self.end_transitions.unsqueeze(0)
        return torch.logsumexp(alpha, dim=1)

    def decode(self, emissions, mask):
        B, T, C = emissions.shape
        viterbi = emissions[:, 0] + self.start_transitions.unsqueeze(0)
        backpointers = []
        for t in range(1, T):
            trans = self.transitions.unsqueeze(0)
            scores = viterbi.unsqueeze(1) + trans
            best_scores, best_ids = scores.max(dim=2)
            emit = emissions[:, t]
            new_viterbi = best_scores + emit
            viterbi = torch.where(mask[:, t].unsqueeze(1), new_viterbi, viterbi)
            backpointers.append(best_ids)
        viterbi = viterbi + self.end_transitions.unsqueeze(0)
        lengths = mask.long().sum(dim=1)
        best_paths = []
        for b in range(B):
            seq_len = lengths[b].item()
            best_last = viterbi[b].argmax().item()
            path = [best_last]
            for t in range(seq_len - 2, -1, -1):
                best_last = backpointers[t][b, best_last].item()
                path.append(best_last)
            path.reverse()
            path = path + [0] * (T - seq_len)
            best_paths.append(path)
        return best_paths


class LegalNER(nn.Module):
    def __init__(self, backbone, num_tags, dropout=0.3, freeze_layers=10):
        super().__init__()
        self.backbone = backbone
        self.num_tags = num_tags
        d_model = backbone.tok_emb.weight.shape[1]
        self.d_model = d_model
        self.freeze_layers = freeze_layers
        self.bilstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model // 2,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
        )
        self.ner_dropout = nn.Dropout(dropout)
        self.ner_head = nn.Linear(d_model, num_tags)
        self.crf = CRF(num_tags)
        nn.init.normal_(self.ner_head.weight, std=0.02)
        nn.init.zeros_(self.ner_head.bias)

        # Freeze the first N backbone layers
        for i, layer in enumerate(self.backbone.layers):
            if i < freeze_layers:
                for p in layer.parameters():
                    p.requires_grad = False
        # Freeze embeddings too
        self.backbone.tok_emb.weight.requires_grad = False
        n_frozen = sum(1 for p in self.backbone.parameters() if not p.requires_grad)
        n_total = sum(1 for p in self.backbone.parameters())
        print(f"Frozen {n_frozen}/{n_total} backbone param tensors (first {freeze_layers}/{len(self.backbone.layers)} layers + embeddings)")

    def get_hidden_states(self, ids, attention_mask=None):
        B, T = ids.shape
        x = self.backbone.drop(self.backbone.tok_emb(ids))
        causal = self.backbone._causal_mask(T, ids.device)
        if attention_mask is not None:
            pad_mask = torch.zeros(B, 1, 1, T, device=ids.device, dtype=x.dtype)
            pad_mask.masked_fill_(attention_mask[:, None, None, :] == 0, float("-inf"))
            mask = causal + pad_mask
        else:
            mask = causal
        for i, layer in enumerate(self.backbone.layers):
            if i < self.freeze_layers:
                # Frozen layers — no grad, no checkpointing needed
                with torch.no_grad():
                    x, _ = layer(x, self.backbone.rope_cos, self.backbone.rope_sin, mask)
                x = x.detach()  # cut the graph here
            else:
                # Use gradient checkpointing for trainable layers
                def create_custom_forward(module):
                    def custom_forward(x, cos, sin, mask):
                        out, _ = module(x, cos, sin, mask)
                        return out
                    return custom_forward
                x = ckpt_fn(create_custom_forward(layer),
                           x, self.backbone.rope_cos, self.backbone.rope_sin, mask,
                           use_reentrant=False)
        x = self.backbone.norm(x)
        return x

    def forward(self, input_ids, labels=None, attention_mask=None):
        hidden = self.get_hidden_states(input_ids, attention_mask=attention_mask)
        hidden, _ = self.bilstm(hidden)
        emissions = self.ner_head(self.ner_dropout(hidden))
        out = {"logits": emissions}
        if labels is not None:
            crf_mask = (labels != -100)
            safe_labels = labels.clone()
            safe_labels[~crf_mask] = 0
            out["loss"] = self.crf(emissions, safe_labels, crf_mask)
        return out

    def decode(self, input_ids, attention_mask=None):
        hidden = self.get_hidden_states(input_ids, attention_mask=attention_mask)
        hidden, _ = self.bilstm(hidden)
        emissions = self.ner_head(hidden)
        crf_mask = attention_mask.bool() if attention_mask is not None else torch.ones(input_ids.shape, dtype=torch.bool, device=input_ids.device)
        return self.crf.decode(emissions, crf_mask)

In [23]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_cosine_mult(step, total_steps, warmup_steps):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * prog))

TRIM_CHARS = set('()[]{},:;/\\\'\"---!?.')

def _trim_spans(doc_spans, text):
    """Strip leading/trailing punctuation from char spans."""
    trimmed = []
    for s_start, s_end, s_label in doc_spans:
        if s_start >= s_end:
            continue
        snippet = text[s_start:s_end]
        l = 0
        while l < len(snippet) and snippet[l] in TRIM_CHARS:
            l += 1
        r = len(snippet)
        while r > l and snippet[r - 1] in TRIM_CHARS:
            r -= 1
        if r > l:
            trimmed.append([s_start + l, s_start + r, s_label])
    return trimmed

def tag_preds_to_char_spans(tag_preds, char_spans, text):
    """BIO: convert tag IDs to char-level spans. B- starts, I-X continues iff same type."""
    doc_spans = []
    current_label = None
    span_start_char = None
    span_end_char = None
    for i, tag_id in enumerate(tag_preds):
        tag_name = id2tag.get(tag_id, "O")
        cs, ce = char_spans[i]
        if tag_name.startswith("B-"):
            if current_label is not None and span_start_char is not None:
                doc_spans.append([span_start_char, span_end_char, current_label])
            current_label = tag_name[2:]
            span_start_char = cs
            span_end_char = ce
        elif tag_name.startswith("I-"):
            if current_label == tag_name[2:]:
                span_end_char = max(span_end_char, ce) if span_end_char else ce
            else:
                if current_label is not None and span_start_char is not None:
                    doc_spans.append([span_start_char, span_end_char, current_label])
                current_label = tag_name[2:]
                span_start_char = cs
                span_end_char = ce
        else:  # O
            if current_label is not None and span_start_char is not None:
                doc_spans.append([span_start_char, span_end_char, current_label])
            current_label = None
            span_start_char = None
            span_end_char = None
    if current_label is not None and span_start_char is not None:
        doc_spans.append([span_start_char, span_end_char, current_label])
    return _trim_spans(doc_spans, text)

In [24]:
def assign_bio_tags(char_spans, gold_annotations, tag2id):
    """BIO tag assignment: B- for first token of entity, I- for continuation."""
    n_tokens = len(char_spans)
    tags = [O_TAG] * n_tokens
    golds = sorted(gold_annotations, key=lambda a: int(a[0]))
    for ann in golds:
        gs, ge, label = int(ann[0]), int(ann[1]), ann[2]
        if label not in LABELS:
            continue
        b_tag = tag2id.get(f"B-{label}", O_TAG)
        i_tag = tag2id.get(f"I-{label}", O_TAG)
        first = True
        for t_idx, (cs, ce) in enumerate(char_spans):
            if ce <= cs:
                continue
            if cs < ge and ce > gs:
                tags[t_idx] = b_tag if first else i_tag
                first = False
    return tags

## 4. Group BPE pieces while preserving character offsets

The original BPE tokenizer still sees the original text. We group its pieces using word and punctuation boundaries that align with BPE token boundaries. If a BPE piece contains both a word and punctuation, that piece stays intact. Both variants use the same windows, which begin and end at group boundaries.

A gold-span round trip on validation shows how many exact spans each representation can recover **when given gold tags**, before a model is involved. This includes the original notebook's boundary trimming.

In [25]:
UNIT_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def word_groups(text, spans, window_size):
    """Return half-open BPE index ranges for words and punctuation."""
    starts = {match.start() for match in UNIT_PATTERN.finditer(text)}
    groups = []
    first = 0
    last_end = None
    for index, (start, end) in enumerate(spans):
        if last_end is not None and start in starts and last_end <= start:
            groups.append((first, index))
            first = index
            last_end = None
        if end > start:
            last_end = end if last_end is None else max(last_end, end)
    if spans:
        groups.append((first, len(spans)))
    return [
        (piece_start, min(piece_start + window_size, end))
        for start, end in groups
        for piece_start in range(start, end, window_size)
    ]

def grouped_char_span(spans, start, end):
    visible = [(left, right) for left, right in spans[start:end] if right > left]
    if not visible:
        return spans[start]
    return min(left for left, _ in visible), max(right for _, right in visible)

def aligned_windows(groups, token_count, window_size, stride):
    """Plan overlapping BPE windows with boundaries between whole groups."""
    if not groups:
        return []
    boundaries = [0] + [end for _, end in groups]
    tail = boundaries[bisect.bisect_left(boundaries, max(0, token_count - window_size))]
    windows = []
    start = 0
    while start < token_count:
        end = boundaries[bisect.bisect_right(boundaries, start + window_size) - 1]
        windows.append((start, end))
        if end == token_count:
            break
        next_start = boundaries[bisect.bisect_left(boundaries, min(start + stride, end))]
        start = min(next_start, tail)
    return windows

@dataclass
class PreparedDocument:
    text: str
    token_ids: list
    bpe_spans: list
    groups: list
    windows: list
    gold: list

def prepare_document(doc_id):
    text = get_doc_text(all_text[doc_id])
    token_ids, spans = encode_with_offsets(tokenizer, text)
    groups = word_groups(text, spans, CFG.window_size)
    windows = aligned_windows(groups, len(token_ids), CFG.window_size, CFG.window_stride)
    return PreparedDocument(text, token_ids, spans, groups, windows, all_gold.get(doc_id, []))

def label_units(document, mode):
    if mode == "subword":
        groups = [(index, index + 1) for index in range(len(document.token_ids))]
        spans = document.bpe_spans
    else:
        groups = document.groups
        spans = [grouped_char_span(document.bpe_spans, *group) for group in groups]
    tags = assign_bio_tags(spans, document.gold, tag2id)
    return groups, spans, tags

documents = {doc_id: prepare_document(doc_id) for doc_id in tqdm(doc_ids, desc="Tokenizing")}

def exact_spans(annotations):
    return {(int(s), int(e), label) for s, e, label, *_ in annotations if label in tag2id_labels}

tag2id_labels = set(LABELS)
for mode in ("subword", "word"):
    total = recovered = 0
    for doc_id in val_ids:
        document = documents[doc_id]
        _, spans, tags = label_units(document, mode)
        gold = exact_spans(document.gold)
        decoded = exact_spans(tag_preds_to_char_spans(tags, spans, document.text))
        recovered += len(gold & decoded)
        total += len(gold)
    print(f"{mode:7s} gold-tag round trip: {recovered}/{total} exact validation spans")

Tokenizing:   0%|          | 0/70 [00:00<?, ?it/s]

subword gold-tag round trip: 299/314 exact validation spans
word    gold-tag round trip: 277/314 exact validation spans


## 5. Make training batches and pool the hidden states

Each BPE position has a group ID. For word BIO, all pieces of the same word have the same group ID. The model adds their hidden vectors and divides by the number of pieces. For the subword baseline each group has one piece. The BiLSTM receives only real tag positions, and the CRF mask excludes padding from the loss.

Rare-window oversampling uses the original subword labels to choose windows, so both runs train on the same windows with the same multiplicities.

In [26]:
def training_windows(document, mode, repeats=None):
    groups, spans, tags = label_units(document, mode)
    group_index = {start: index for index, (start, _) in enumerate(groups)}
    group_index[len(document.token_ids)] = len(groups)
    examples = []
    for index, (start, end) in enumerate(document.windows):
        lo, hi = group_index[start], group_index[end]
        group_sizes = [right - left for left, right in groups[lo:hi]]
        examples.extend([(
            document.token_ids[start:end], tags[lo:hi], group_sizes, lo
        )] * (1 if repeats is None else repeats[index]))
    return spans, examples

def rare_window_repeats(document):
    _, examples = training_windows(document, "subword")
    rare_types = ("WIT", "RESP", "APP", "R.COUNSEL", "A.COUNSEL", "JUDGE")
    rare_tags = {tag2id[prefix + label] for prefix in ("B-", "I-")
                 for label in rare_types if prefix + label in tag2id}
    witness_tags = {tag2id[tag] for tag in ("B-WIT", "I-WIT") if tag in tag2id}
    return [
        21 if set(tags) & witness_tags else 1 + 5 * len(set(tags) & rare_tags)
        for _, tags, _, _ in examples
    ]

repeats_by_doc = {
    doc_id: rare_window_repeats(documents[doc_id]) for doc_id in train_ids
} if OVERSAMPLE_RARE_WINDOWS else {}

def collate_windows(examples):
    batch_size = len(examples)
    lengths = torch.tensor([len(tags) for _, tags, _, _ in examples])
    ids = torch.zeros(batch_size, max(len(tokens) for tokens, *_ in examples), dtype=torch.long)
    labels = torch.full((batch_size, int(lengths.max())), -100, dtype=torch.long)
    group_ids = torch.full_like(ids, -1)
    for row, (tokens, tags, sizes, _) in enumerate(examples):
        ids[row, :len(tokens)] = torch.tensor(tokens)
        labels[row, :len(tags)] = torch.tensor(tags)
        group_ids[row, :len(tokens)] = torch.repeat_interleave(torch.arange(len(tags)), torch.tensor(sizes))
    return ids, group_ids, labels, lengths

class PooledLegalNER(LegalNER):
    def forward(self, input_ids, group_ids, unit_lengths, unit_count):
        mask = group_ids.ge(0)
        hidden = self.get_hidden_states(input_ids, attention_mask=mask)
        index = group_ids.clamp_min(0).unsqueeze(-1)
        sums = hidden.new_zeros((hidden.shape[0], unit_count, hidden.shape[-1]), dtype=torch.float32)
        sums.scatter_add_(1, index.expand_as(hidden), hidden.float() * mask.unsqueeze(-1))
        counts = sums.new_zeros((hidden.shape[0], unit_count, 1))
        counts.scatter_add_(1, index, mask.unsqueeze(-1).float())
        pooled = (sums / counts.clamp_min(1)).to(hidden.dtype)

        packed = nn.utils.rnn.pack_padded_sequence(
            pooled, unit_lengths, batch_first=True, enforce_sorted=False
        )
        packed, _ = self.bilstm(packed)
        contextual, _ = nn.utils.rnn.pad_packed_sequence(
            packed, batch_first=True, total_length=unit_count
        )
        return self.ner_head(self.ner_dropout(contextual)).float()

## 6. Train and compare strict span F1

Each arm gets a fresh copy of the same decoder checkpoint. Both use the original run's frozen layers, learning rates, gradient accumulation, and early stopping. During validation, a unit keeps the scores from the window where the whole unit is farthest from the nearest BPE edge. The CRF then decodes once per document.

Use the per-label results and the gold-tag round trip alongside macro-F1. The three-document split makes this a small local comparison.

In [27]:
@torch.no_grad()
def evaluate(model, mode):
    model.eval()
    predicted = defaultdict(set)
    gold = defaultdict(set)

    for doc_id in val_ids:
        document = documents[doc_id]
        spans, examples = training_windows(document, mode)
        for start, end, label in exact_spans(document.gold):
            gold[label].add((doc_id, start, end))
        if not spans:
            continue
        best_distance = [-1] * len(spans)
        selected = torch.zeros(len(spans), CFG.bio_num_tags)
        for first in range(0, len(examples), CFG.batch_size):
            batch = examples[first:first + CFG.batch_size]
            ids, groups, labels, lengths = collate_windows(batch)
            with torch.amp.autocast(DEVICE.type, enabled=CFG.use_amp):
                logits = model(ids.to(DEVICE), groups.to(DEVICE), lengths, labels.size(1)).cpu()
            for row, (tokens, _, sizes, start_unit) in enumerate(batch):
                local_start = 0
                for offset, size in enumerate(sizes):
                    unit = start_unit + offset
                    local_end = local_start + size
                    distance = min(local_start, len(tokens) - local_end)
                    if distance > best_distance[unit]:
                        selected[unit] = logits[row, offset]
                        best_distance[unit] = distance
                    local_start = local_end

        emission = selected.unsqueeze(0).to(DEVICE)
        mask = torch.ones(1, len(spans), dtype=torch.bool, device=DEVICE)
        tags = model.crf.decode(emission, mask)[0]
        for start, end, label in tag_preds_to_char_spans(tags, spans, document.text):
            predicted[label].add((doc_id, start, end))

    per_label = {}
    for label in LABELS:
        n_pred, n_gold = len(predicted[label]), len(gold[label])
        tp = len(predicted[label] & gold[label])
        per_label[label] = 2 * tp / (n_pred + n_gold) if n_pred + n_gold else 0.0
    return sum(per_label.values()) / len(LABELS), per_label

def run_experiment(mode):
    set_seed(CFG.seed)
    examples = [
        example
        for doc_id in train_ids
        for example in training_windows(documents[doc_id], mode, repeats_by_doc.get(doc_id))[1]
    ]
    loader = DataLoader(
        examples, batch_size=CFG.batch_size, shuffle=True, drop_last=True,
        collate_fn=collate_windows, generator=torch.Generator().manual_seed(CFG.seed),
    )
    if not len(loader):
        raise ValueError("No complete training batch. Check the data or lower CFG.batch_size.")
    backbone = load_phase1_backbone(CFG, PHASE1_CKPT, DEVICE)
    model = PooledLegalNER(backbone, CFG.bio_num_tags, dropout=0.3, freeze_layers=14).to(DEVICE)
    backbone_params, head_params = [], []
    for name, param in model.named_parameters():
        if param.requires_grad:
            (backbone_params if name.startswith("backbone.") else head_params).append(param)
    base_lrs = (5e-6, 5e-4)
    optimizer = torch.optim.AdamW([
        {"params": backbone_params, "lr": base_lrs[0]},
        {"params": head_params, "lr": base_lrs[1]},
    ], weight_decay=CFG.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=CFG.use_amp)
    updates_per_epoch = math.ceil(len(loader) / CFG.grad_accum)
    total_updates = updates_per_epoch * EPOCHS
    best = {"macro_f1": -1.0}
    steps_since_best = update = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        progress = tqdm(loader, desc=f"{mode} epoch {epoch}")
        for index, (ids, groups, labels, lengths) in enumerate(progress):
            ids, groups, labels = ids.to(DEVICE), groups.to(DEVICE), labels.to(DEVICE)
            group_start = index - index % CFG.grad_accum
            accum_size = min(CFG.grad_accum, len(loader) - group_start)
            mask = labels.ne(-100)
            with torch.amp.autocast(DEVICE.type, enabled=CFG.use_amp):
                logits = model(ids, groups, lengths, labels.size(1))
                loss = model.crf(logits, labels.masked_fill(~mask, 0), mask)
            scaler.scale(loss / accum_size).backward()
            if index - group_start + 1 == accum_size:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(backbone_params + head_params, CFG.max_grad_norm)
                multiplier = get_cosine_mult(update, total_updates, CFG.warmup_steps)
                for group, base_lr in zip(optimizer.param_groups, base_lrs):
                    group["lr"] = max(CFG.min_lr, multiplier * base_lr)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                update += 1
                if update % 50 == 0:
                    progress.set_postfix(loss=f"{loss.item():.3f}")

        macro_f1, per_label = evaluate(model, mode)
        print(f"{mode} epoch {epoch}: strict document macro-F1 = {macro_f1:.4f}")
        if macro_f1 > best["macro_f1"]:
            best = {"epoch": epoch, "macro_f1": macro_f1, "per_label": per_label}
            torch.save({"model": model.state_dict(), "mode": mode, **best}, RESULTS_DIR / f"{mode}_best.pt")
            steps_since_best = 0
        else:
            steps_since_best += 1
            if steps_since_best >= EARLY_STOP_PATIENCE:
                break
    return best

In [28]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results = {}
for mode in RUN_MODES:
    if mode not in ("subword", "word"):
        raise ValueError(f"Unknown tagging mode: {mode}")
    results[mode] = run_experiment(mode)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

for mode, result in results.items():
    print(f"{mode:7s} best epoch {result['epoch']}: macro-F1 {result['macro_f1']:.4f}")
    for label in LABELS:
        print(f"  {label:12s} {result['per_label'][label]:.4f}")

if {"subword", "word"} <= results.keys():
    delta = results["word"]["macro_f1"] - results["subword"]["macro_f1"]
    print(f"Word minus subword macro-F1: {delta:+.4f}")

(RESULTS_DIR / "validation_results.json").write_text(
    json.dumps({"validation_ids": val_ids, "results": results}, indent=2),
    encoding="utf-8",
)

Phase 1 backbone loaded from best.pt
  Parameters: 150,287,616
Frozen 127/173 backbone param tensors (first 14/19 layers + embeddings)


word epoch 1:   0%|          | 0/1520 [00:00<?, ?it/s]

word epoch 1: strict document macro-F1 = 0.4800


word epoch 2:   0%|          | 0/1520 [00:00<?, ?it/s]

word epoch 2: strict document macro-F1 = 0.5037


word epoch 3:   0%|          | 0/1520 [00:00<?, ?it/s]

word epoch 3: strict document macro-F1 = 0.5001


word epoch 4:   0%|          | 0/1520 [00:00<?, ?it/s]

word epoch 4: strict document macro-F1 = 0.4666
word    best epoch 2: macro-F1 0.5037
  APP          0.3158
  RESP         0.0000
  A.COUNSEL    0.5161
  R.COUNSEL    0.6207
  JUDGE        0.5714
  WIT          0.0000
  AUTH         0.6465
  COURT        0.8451
  STAT         0.8052
  PREC         0.2439
  DATE         0.7805
  CASENO       0.6988


616